In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from scipy.linalg import rq
from scipy.optimize import least_squares
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

from IPython.display import Image as ShowImage, display
from PIL import Image

def save_gif(frames, name, duration=80):
    # write an animation (list of uint8 RGB frames) as a GIF and show it
    path = f'figures/{name}.gif' if SAVE_FIGURES else f'{name}.gif'
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    gif = [Image.fromarray(f) for f in frames]
    palette = gif[0].quantize(192)                       # one palette, no dithering: a small file
    gif = [g.quantize(palette=palette, dither=Image.Dither.NONE) for g in gif]
    gif[0].save(path, save_all=True, append_images=gif[1:], duration=duration, loop=0, optimize=True)
    display(ShowImage(filename=path))

In [ ]:
%%sh
mkdir -p 14
wget --no-verbose --output-document=14/texture_facade1.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_facade1.jpg
wget --no-verbose --output-document=14/texture_facade2.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_facade2.jpg
wget --no-verbose --output-document=14/texture_facade3.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_facade3.jpg
wget --no-verbose --output-document=14/texture_road.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_road.jpg
wget --no-verbose --output-document=14/texture_paving.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_paving.jpg
wget --no-verbose --output-document=14/texture_bricks.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_bricks.jpg

# Lecture 14 - Camera Models

A pinhole camera in a few lines of numpy: perspective vs. orthographic projection, the dolly zoom, vanishing points with homogeneous coordinates, what each camera parameter does to the image, radial lens distortion, and camera calibration with the linear (DLT) and non-linear method, and with OpenCV from photos of a checkerboard.

To see what the camera does we render images with a tiny **ray caster**: one ray per pixel, sent from the camera centre through that pixel ($K^{-1}\mathbf{x}$, rotated into the world), coloured by the first surface it hits. Projection and rendering are the same geometry run in opposite directions.

The textures of the rendered street are photos from [ambientCG](https://ambientcg.com) (CC0). No GPU needed.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)).

## A pinhole camera

A 3D point $\mathbf{X}$ (world coordinates) is mapped to the image by $\mathbf{x} \sim K [R \mid \mathbf{t}] \mathbf{X}$: first into camera coordinates ($R$, $\mathbf{t}$, the *extrinsics*), then onto the image plane and into pixels ($K$, the *intrinsics*), and finally we divide by the third (homogeneous) coordinate.

In [ ]:
def intrinsics(f, cx=320, cy=240):
    return np.array([[f, 0, cx], [0, f, cy], [0, 0, 1.0]])

def look_at(camera_center, target, up=(0, 1, 0)):
    """rotation R and translation t of a camera at `camera_center` looking at `target` (x right, y down, z forward)"""
    z = np.asarray(target, float) - camera_center
    z /= np.linalg.norm(z)
    x = np.cross(z, up)
    x /= np.linalg.norm(x)
    y = np.cross(z, x)
    R = np.stack([x, y, z])
    return R, -R @ camera_center

def project(X, K, R, t):
    """perspective projection of (N, 3) world points to (N, 2) pixels"""
    x = (K @ (R @ X.T + t[:, None])).T
    return x[:, :2] / x[:, 2:]

def project_orthographic(X, K, R, t, scale):
    """orthographic projection: drop the depth instead of dividing by it"""
    Xc = (R @ X.T + t[:, None]).T
    return Xc[:, :2] * scale + K[:2, 2]

# a small scene: a row of identical houses along a street (y is up in the world)
def house(x0, z0, w=2.0, h=2.0, d=3.0):
    v = np.array([[0, 0, 0], [w, 0, 0], [w, 0, d], [0, 0, d], [0, h, 0], [w, h, 0], [w, h, d], [0, h, d], [w/2, h*1.5, 0], [w/2, h*1.5, d]])
    edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7),(4,8),(5,8),(7,9),(6,9),(8,9)]
    return v + [x0, 0, z0], edges

houses = [house(-4, z) for z in range(0, 40, 5)] + [house(2, z) for z in range(0, 40, 5)]
X = np.concatenate([v for v, _ in houses])
edges = [(a + 10*i, b + 10*i) for i, (_, e) in enumerate(houses) for a, b in e]

def draw(ax, x, title):
    for a, b in edges:
        ax.plot(x[[a, b], 0], x[[a, b], 1], 'k-', linewidth=1)
    ax.set_xlim(0, 640)
    ax.set_ylim(480, 0)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

C = np.array([0.0, 1.6, -6.0])               # eye height 1.6 m, 6 m in front of the first houses
R, t = look_at(C, [0, 1.6, 10])
K = intrinsics(300)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
draw(axs[0], project(X, K, R, t), 'Perspective projection')
draw(axs[1], project_orthographic(X, K, R, t, scale=40), 'Orthographic projection')
savefig('camera_perspective_orthographic')
plt.show()

## Rendering images: a tiny ray caster

Projection maps a 3D point to a pixel. Rendering goes the other way: for every pixel $\mathbf{x}$ the camera centre $\mathbf{C}$ and the direction $R^\top K^{-1} \mathbf{x}$ define a ray, and the pixel gets the colour of the first surface along that ray. An **orthographic** camera has parallel rays instead (all along the optical axis, starting on a plane), and **radial distortion** bends the rays of the pixels away from the centre. The scene: a street of textured boxes (houses) and a few spheres. Textures are filtered as on a GPU (mipmaps) and every pixel averages 3 x 3 rays, against aliasing.

In [ ]:
def camera_rays(K, R, t, width, height, k1=0.0, ortho=None):
    # one ray per pixel: origins and unit directions in world coordinates, shape (H, W, 3);
    # k1: radial distortion; ortho: pixels per metre of an orthographic camera (parallel rays)
    u, v = np.meshgrid(np.arange(width) + 0.5, np.arange(height) + 0.5)
    xn = (u - K[0, 2]) / K[0, 0]                     # K^-1 x: normalised image coordinates
    yn = (v - K[1, 2]) / K[1, 1]
    valid = np.ones_like(xn, bool)
    if k1:                                           # undo x_d = x (1 + k1 r^2) by fixed-point iteration
        xu, yu = xn.copy(), yn.copy()
        for _ in range(50):
            s = 1 + k1 * (xu ** 2 + yu ** 2)
            xu, yu = xn / s, yn / s
        s = 1 + k1 * (xu ** 2 + yu ** 2)
        valid = (np.abs(xu * s - xn) + np.abs(yu * s - yn) < 1e-4) & (s > 0)   # pixels no ray maps to stay black
        xn, yn = xu, yu
    C = -R.T @ t                                     # camera centre in world coordinates
    if ortho:
        offsets = np.stack([(u - K[0, 2]) / ortho, (v - K[1, 2]) / ortho, np.zeros_like(u)], -1)
        origins = C + offsets @ R - 1000 * R[2]      # rays start on a plane (far behind C, so nothing is cut off) ...
        dirs = np.broadcast_to(R[2], origins.shape).copy()   # ... and all point along the optical axis
    else:
        dirs = np.stack([xn, yn, np.ones_like(xn)], -1) @ R   # R^T (x_n, y_n, 1): through the camera centre
        dirs /= np.linalg.norm(dirs, axis=-1, keepdims=True)
        origins = np.broadcast_to(C, dirs.shape).copy()
    return origins, dirs, valid

LIGHT = np.array([0.4, 1.0, -0.6]) / np.linalg.norm([0.4, 1.0, -0.6])

def texture(name, width, height):
    # a photo texture (ambientCG, CC0) that repeats every width x height metres; s, t in metres (t up).
    # Far away (or seen at a grazing angle) one pixel covers many texels: reading a single texel would alias
    # (shimmering, moire). Like a GPU we keep a "mipmap", the photo at half, quarter, ... resolution, and read
    # the level whose texels are about as large as the pixel's footprint (bilinear, blended between two levels).
    levels = [plt.imread(f'14/texture_{name}.jpg') / 255.0]
    while min(levels[-1].shape[:2]) > 4:
        levels.append(cv2.resize(levels[-1], None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA))
    texels_per_metre = max(levels[0].shape[1] / width, levels[0].shape[0] / height)
    def bilinear(image, s, t):
        h, w = image.shape[:2]
        x, y = np.mod(s / width, 1) * w - 0.5, (1 - np.mod(t / height, 1)) * h - 0.5
        x0, y0 = np.floor(x).astype(int), np.floor(y).astype(int)
        fx, fy = (x - x0)[:, None], (y - y0)[:, None]
        x0, x1, y0, y1 = x0 % w, (x0 + 1) % w, y0 % h, (y0 + 1) % h
        return (1 - fy) * ((1 - fx) * image[y0, x0] + fx * image[y0, x1]) + fy * ((1 - fx) * image[y1, x0] + fx * image[y1, x1])
    def lookup(s, t, footprint):
        # footprint: size of the pixel on the surface (metres)
        lod = np.clip(np.log2(np.maximum(footprint * texels_per_metre, 1)), 0, len(levels) - 1)
        level, blend = np.floor(lod).astype(int), (lod - np.floor(lod))[:, None]
        out = np.zeros((len(s), 3))
        for l in np.unique(level):
            m = level == l
            fine, coarse = bilinear(levels[l], s[m], t[m]), bilinear(levels[min(l + 1, len(levels) - 1)], s[m], t[m])
            out[m] = (1 - blend[m]) * fine + blend[m] * coarse
        return out
    return lookup

def render(scene, origins, dirs, fog=400.0, pixel_angle=0.0, pixel_size=0.0):
    # colour of the nearest surface along every ray (a z-buffer over the objects). A pixel covers
    # pixel_size + distance * pixel_angle metres across the ray (perspective: angle 1/f, orthographic: size 1/scale);
    # on a surface tilted away from the ray it covers more, 1/sqrt(cos) is a compromise between its two sides
    H, W = dirs.shape[:2]
    footprint = lambda dist, cos: (pixel_size + dist * pixel_angle) / np.sqrt(np.maximum(np.abs(cos), 0.02))
    o, d = origins.reshape(-1, 3), dirs.reshape(-1, 3)
    best = np.full(len(d), np.inf)
    up = np.clip(d[:, 1:2], 0, 1)
    sky = np.clip([0.62, 0.78, 0.95] * (0.8 + 0.2 * up) + [0.15, 0.1, 0.0] * (1 - np.clip(4 * up, 0, 1)), 0, 1)
    col = sky.copy()
    for obj in scene:
        if obj[0] == 'rect':                         # textured rectangle: corner p0, edges e1, e2
            _, p0, e1, e2, texture = obj
            n = np.cross(e1, e2)
            n /= np.linalg.norm(n)
            with np.errstate(divide='ignore', invalid='ignore'):
                tt = ((p0 - o) @ n) / (d @ n)
            p = o + tt[:, None] * d
            a, b = (p - p0) @ e1 / (e1 @ e1), (p - p0) @ e2 / (e2 @ e2)
            hit = (tt > 1e-6) & (tt < best) & (a >= 0) & (a <= 1) & (b >= 0) & (b <= 1)
            col[hit] = texture(a[hit] * np.linalg.norm(e1), b[hit] * np.linalg.norm(e2),
                               footprint(tt[hit], d[hit] @ n)) * (0.7 + 0.3 * abs(n @ LIGHT))
            best[hit] = tt[hit]
        elif obj[0] == 'box':                        # axis-aligned box lo..hi: textured walls, flat roof
            _, lo, hi, wall, roof = obj
            oc = o - (lo + hi) / 2                     # only test the rays that pass through the box's bounding sphere
            b = np.einsum('ij,ij->i', oc, d)
            rays = np.flatnonzero(b ** 2 - np.einsum('ij,ij->i', oc, oc) + ((hi - lo) ** 2).sum() / 4 > 0)
            with np.errstate(divide='ignore', invalid='ignore'):
                t1, t2 = (lo - o[rays]) / d[rays], (hi - o[rays]) / d[rays]
            tnear, axis = np.minimum(t1, t2).max(1), np.minimum(t1, t2).argmax(1)
            front = (tnear <= np.maximum(t1, t2).min(1)) & (tnear > 1e-6) & (tnear < best[rays])
            hit, tnear, ax = rays[front], tnear[front], axis[front]
            p = o[hit] + tnear[:, None] * d[hit]
            c = np.tile(np.asarray(roof, float), (len(hit), 1))
            fp = footprint(tnear, d[hit, ax])
            c[ax == 0] = wall(p[ax == 0, 2] - lo[2], p[ax == 0, 1] - lo[1], fp[ax == 0])
            c[ax == 2] = wall(p[ax == 2, 0] - lo[0], p[ax == 2, 1] - lo[1], fp[ax == 2])
            col[hit] = c * (0.55 + 0.45 * np.abs(LIGHT[ax]))[:, None]
            best[hit] = tnear
        elif obj[0] == 'sphere':                     # a beach ball
            _, centre, radius, colours = obj
            oc = o - centre
            b = (oc * d).sum(1)
            disc = b ** 2 - (oc ** 2).sum(1) + radius ** 2
            tt = -b - np.sqrt(np.maximum(disc, 0))
            hit = (disc > 0) & (tt > 1e-6) & (tt < best)
            nrm = (o[hit] + tt[hit, None] * d[hit] - centre) / radius
            segment = np.floor((np.arctan2(nrm[:, 0], nrm[:, 2]) / np.pi + 1) * len(colours) / 2).astype(int) % len(colours)
            c = np.array(colours, float)[segment]
            c[np.abs(nrm[:, 1]) > 0.93] = 0.95
            lambert = np.clip(nrm @ LIGHT, 0, 1)
            col[hit] = np.clip(c * (0.35 + 0.65 * lambert)[:, None] + 0.4 * lambert[:, None] ** 40, 0, 1)
            best[hit] = tt[hit]
    if fog:                                          # far away things fade into the sky
        f = np.exp(-np.minimum(best, 1e6) / fog)[:, None]
        col = f * col + (1 - f) * sky
    return col.reshape(H, W, 3)

def photo(scene, K, R, t, width=640, height=480, k1=0.0, ortho=None, fog=400.0, ss=3):
    # render with ss x ss rays per pixel (anti-aliasing of the edges), averaged; uint8 RGB
    Ks = K.copy()
    Ks[:2] *= ss
    o, d, valid = camera_rays(Ks, R, t, width * ss, height * ss, k1, ortho * ss if ortho else None)
    pixel = dict(pixel_size=1 / (ortho * ss)) if ortho else dict(pixel_angle=1 / Ks[0, 0])   # footprint of one ray
    img = render(scene, o, d, fog, **pixel) * valid[..., None]
    return cv2.resize((img * 255).astype(np.uint8), (width, height), interpolation=cv2.INTER_AREA)

BALL = [(0.9, 0.2, 0.2), (0.95, 0.85, 0.2), (0.2, 0.5, 0.9), (0.95, 0.95, 0.95), (0.2, 0.7, 0.3), (0.95, 0.55, 0.1)]
FACADES = [texture(f'facade{i}', 13, 13) for i in (1, 2, 3)]    # 13 m x 13 m per copy of the photo
ROAD, PAVING, BRICKS = texture('road', 7.5, 7.5), texture('paving', 1.15, 1.15), texture('bricks', 2.4, 1.2)
ROAD_W, GAP, BLDG = 7.5, 1.5, (10.0, 13.0, 12.0)      # road width, pavement, building width x height x depth (m)
X_FACE = ROAD_W / 2 + GAP                             # the buildings' street-facing walls are at x = +-6

def street_scene(n=6):
    # a straight street along z: a 7.5 m road, paved pavements, and n buildings on either side (every 15 m)
    big = 1000.0
    scene = [('rect', np.array([-ROAD_W / 2, 0, -big]), np.array([ROAD_W, 0, 0]), np.array([0, 0, 2 * big]), ROAD),
             ('rect', np.array([-big, 0, -big]), np.array([big - ROAD_W / 2, 0, 0]), np.array([0, 0, 2 * big]), PAVING),
             ('rect', np.array([ROAD_W / 2, 0, -big]), np.array([big, 0, 0]), np.array([0, 0, 2 * big]), PAVING)]
    w, h, d = BLDG
    for i, (x0, x1) in enumerate([(-X_FACE - w, -X_FACE), (X_FACE, X_FACE + w)]):
        for j in range(n):
            z0 = 15.0 * j
            scene.append(('box', np.array([x0, 0, z0]), np.array([x1, h, z0 + d]), FACADES[(i + j) % 3], (0.45, 0.45, 0.47)))
    return scene

def label(img, text, pos=(10, 25), scale=0.6):
    img = img.copy()
    cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 4, cv2.LINE_AA)
    cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 1, cv2.LINE_AA)
    return img

street = street_scene()
C = np.array([45.0, 35.0, -30.0])                     # above and to the side of the street
R, t = look_at(C, [0, 0, 40])
perspective = photo(street, intrinsics(380), R, t, fog=None)
orthographic = photo(street, intrinsics(380), R, t, ortho=4.5, fog=None)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for ax, im, title in zip(axs, [perspective, orthographic], ['Perspective: parallel edges converge, far buildings are smaller',
                                                         'Orthographic: parallel edges stay parallel, all buildings the same size']):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')
savefig('camera_render_persp_ortho')
plt.show()

## Single-view ambiguity

A sphere twice as far away and twice as large covers exactly the same pixels: from one image we cannot tell size from distance. On the left, the scene from the side; on the right, what the camera sees.

In [ ]:
wall = ('rect', np.array([-30, -20, 40.0]), np.array([60.0, 0, 0]), np.array([0, 40.0, 0]), BRICKS)
R0, t0 = look_at(np.zeros(3), [0, 0, 1])
K0 = intrinsics(300, 160, 120)
ray = np.array([0.15, 0.0, 1.0])                       # the sphere's centre moves along this ray
frames = []
for s in np.concatenate([np.geomspace(3, 15, 30), np.geomspace(15, 3, 30)]):
    image = photo([wall, ('sphere', s * ray, 0.25 * s, BALL)], K0, R0, t0, 320, 240)
    fig, ax = plt.subplots(figsize=(4.8, 2.4), dpi=100)
    ax.plot([0, 42], [0, 42 * 0.4], 'k:', lw=0.8)
    ax.plot([0, 42], [0, -42 * 0.4], 'k:', lw=0.8)
    ax.plot([40, 40], [-17, 17], color='0.5', lw=3)
    for s2, alpha in [(3, 0.25), (15, 0.25), (s, 1.0)]:
        ax.add_patch(plt.Circle((s2, 0.15 * s2), 0.25 * s2, color='C3', alpha=alpha))
    beta = np.arcsin(0.25 / np.hypot(1, 0.15))           # the cone of rays that touch the sphere
    for a in (np.arctan(0.15) + beta, np.arctan(0.15) - beta):
        ax.plot([0, 40], [0, 40 * np.tan(a)], 'C3-', lw=1)
    ax.plot(0, 0, 'ko')
    ax.text(0.5, 7, 'camera', fontsize=9)
    ax.text(41, -16, 'wall', fontsize=9, ha='right')
    ax.text(0, -16, f'distance {s * np.hypot(1, .15):.1f} m, radius {0.25 * s:.2f} m', fontsize=9)
    ax.set_xlim(-2, 42)
    ax.set_ylim(-17, 17)
    ax.set_aspect('equal')
    ax.axis('off')
    fig.tight_layout(pad=0.1)
    fig.canvas.draw()
    side = np.asarray(fig.canvas.buffer_rgba())[..., :3]
    plt.close(fig)
    side = cv2.resize(side, (480, 240))
    frames.append(np.concatenate([side, np.full((240, 10, 3), 255, np.uint8), label(image, 'camera image')], 1))
save_gif(frames, 'camera_ambiguity', duration=70)

## Spheres in a wide-angle image

A sphere's image is a circle only when the sphere is on the optical axis. Off the axis, the cone of rays that touches the sphere cuts the image plane at a slant, and the image is an ellipse, stretched away from the centre. With a wide-angle lens this is very visible, while straight lines (the brick wall) stay straight.

In [ ]:
spheres = [('rect', np.array([-30, -20, 9.0]), np.array([60.0, 0, 0]), np.array([0, 40.0, 0]), BRICKS)]
spheres += [('sphere', np.array([X, Y, 5.0]), 0.9, BALL) for X in np.arange(-6, 7, 3) for Y in (-2.5, 0, 2.5)]
f_wide = 200                                          # 640 px wide: a horizontal field of view of 2 atan(320 / 200) = 116 degrees
image = photo(spheres, intrinsics(f_wide), R0, t0)
plt.figure(figsize=(10, 7.5))
plt.imshow(image)
plt.title(f'15 identical spheres, f = {f_wide} px (field of view {2 * np.degrees(np.arctan(320 / f_wide)):.0f} degrees)')
plt.axis('off')
savefig('camera_spheres')
plt.show()

## The dolly zoom

Move the camera away from the scene and zoom in (increase $f$) so that the nearest house keeps its size in the image. The further away the camera, the smaller the *relative* depth differences, and the closer the perspective image gets to an orthographic one. (Hitchcock used this in *Vertigo*.)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

distances = np.geomspace(4, 200, 60)
def dolly_frame(ax, distance):
    C = np.array([0.0, 1.6, -distance])
    R, t = look_at(C, [0, 1.6, 10])
    f = 300 * distance / 6                   # f / distance stays constant: the front houses keep their size
    ax.clear()
    draw(ax, project(X, intrinsics(f), R, t), f'camera {distance:.0f} m away, f = {f:.0f} px')

fig, axs = plt.subplots(1, 4, figsize=(24, 5))
for ax, d in zip(axs, [4, 10, 40, 200]):
    dolly_frame(ax, d)
savefig('camera_dolly_zoom')
plt.show()

fig, ax = plt.subplots(figsize=(7, 5.5))
anim = FuncAnimation(fig, lambda i: dolly_frame(ax, np.concatenate([distances, distances[::-1]])[i]), frames=2 * len(distances), interval=60)
plt.close()
HTML(anim.to_jshtml())


# the same with the rendered street: the front buildings keep their size, the street behind them shrinks
frames = []
for d in np.concatenate([np.geomspace(6, 200, 40), np.geomspace(200, 6, 40)]):
    C = np.array([0.0, 1.6, -d])
    R, t = look_at(C, [0, 1.6, 10])
    f = 240 * d / 12                                  # f / d constant: the front buildings keep their size
    frames.append(label(photo(street_scene(12), intrinsics(f, 240, 180), R, t, 480, 360, fog=None),
                        f'camera {d:.0f} m away, f = {f:.0f} px'))
save_gif(frames, 'camera_dolly_zoom', duration=80)

## Vanishing points

Parallel 3D lines with direction $\mathbf{d}$ meet in the image at the vanishing point $\mathbf{v} \sim K R \mathbf{d}$ (the projection of the point at infinity $(\mathbf{d}, 0)$). With homogeneous coordinates (Lecture 5) we can also find it from the image alone: the line through two image points is $\mathbf{l} = \mathbf{x}_1 \times \mathbf{x}_2$, and two lines intersect at $\mathbf{l}_1 \times \mathbf{l}_2$.

In [ ]:
C = np.array([-1.0, 1.6, -6.0])
R, t = look_at(C, [3, 0.5, 10])          # look a bit to the side and down: the street is not parallel to the image plane
K = intrinsics(300)
x = project(X, K, R, t)
hom = lambda p: np.array([p[0], p[1], 1.0])

# the long edges of the houses (along z) and the roof ridges are parallel: direction (0, 0, 1)
parallel = [(a, b) for a, b in edges if np.allclose(X[b] - X[a], [0, 0, 3])]
lines = [np.cross(hom(x[a]), hom(x[b])) for a, b in parallel]
v_from_image = np.cross(lines[0], lines[5])
v_from_image = v_from_image[:2] / v_from_image[2]
v_true = K @ R @ np.array([0, 0, 1.0])
v_true = v_true[:2] / v_true[2]
print('vanishing point from two image lines:', v_from_image.round(2))
print('vanishing point K R d:              ', v_true.round(2))

# vanishing points of all horizontal directions lie on one line: the horizon
d1, d2 = np.array([1.0, 0, 0]), np.array([0, 0, 1.0])
horizon = np.cross(K @ R @ d1, K @ R @ d2)

fig, ax = plt.subplots(figsize=(12, 8))
draw(ax, x, 'Parallel lines meet at the vanishing point; horizontal directions meet on the horizon')
for a, b in parallel:
    ax.plot([x[a, 0], v_true[0]], [x[a, 1], v_true[1]], 'r-', linewidth=0.5, alpha=0.6)
ax.plot(*v_true, 'ro', markersize=10)
xs = np.array([-2000, 3000])
ax.plot(xs, -(horizon[0] * xs + horizon[2]) / horizon[1], 'b--', label='horizon')
ax.set_xlim(min(0, v_true[0] - 50), max(640, v_true[0] + 50))
ax.set_ylim(480, min(0, v_true[1] - 50))
ax.legend()
savefig('camera_vanishing_point')
plt.show()

### Moving the camera does not move the vanishing point, turning it does

Moving the camera ($\mathbf{t}$) leaves the vanishing point $\mathbf{v} \sim K R \mathbf{d}$ and the horizon where they are: they depend only on the direction $\mathbf{d}$ of the lines and on the camera's orientation. Turning the camera ($R$) moves them.

In [ ]:
K = intrinsics(240, 240, 180)
street = street_scene(14)
start = np.array([-1.5, 1.6, -12.0])
frames = []
moves = [(start + [3 * np.sin(a), 4 * np.sin(a / 2) ** 2, 12 * (1 - np.cos(a))], 0.0, 0.0)     # walk, step aside, climb
         for a in np.linspace(0, 2 * np.pi, 48, endpoint=False)]
turns = [(start, np.radians(20 * np.sin(a)), np.radians(8 * np.sin(2 * a)))                       # pan and tilt
         for a in np.linspace(0, 2 * np.pi, 48, endpoint=False)]
for i, (C, yaw, pitch) in enumerate(moves + turns):
    R, t = look_at(C, C + [np.sin(yaw) * np.cos(pitch), np.sin(pitch), np.cos(yaw) * np.cos(pitch)])
    image = photo(street, K, R, t, 480, 360)
    v = K @ R @ np.array([0, 0, 1.0])                 # vanishing point of the street direction
    v = v[:2] / v[2]
    horizon = np.cross(K @ R @ [1.0, 0, 0], K @ R @ [0, 0, 1.0])
    y_at = lambda x: -(horizon[0] * x + horizon[2]) / horizon[1]
    cv2.line(image, (0, int(y_at(0))), (480, int(y_at(480))), (40, 40, 220), 2, cv2.LINE_AA)
    for x0 in (-X_FACE, X_FACE):                       # the edges of the buildings along the street, from 5 m ahead of the camera
        for y in (0, BLDG[1]):
            p = project(np.array([[x0, y, max(C[2] + 5, 0)]]), K, R, t)[0]
            cv2.line(image, tuple(np.int32(p)), tuple(np.int32(v)), (220, 40, 40), 2, cv2.LINE_AA)
    cv2.circle(image, tuple(np.int32(v)), 6, (220, 40, 40), -1, cv2.LINE_AA)
    frames.append(label(image, 'camera moves: vanishing point stays' if i < len(moves) else 'camera turns: vanishing point moves'))
save_gif(frames, 'camera_vanishing_pan', duration=90)

## What each camera parameter does

The intrinsics $K = \begin{bmatrix} \alpha_x & 0 & p_x \\ 0 & \alpha_y & p_y \\ 0 & 0 & 1 \end{bmatrix}$ act on the image only: the focal length zooms, the principal point shifts the image, and different pixel scales $\alpha_x \neq \alpha_y$ stretch it. None of them changes *which* rays the camera sees relative to each other, so the perspective (what occludes what, where lines meet in 3D) stays the same. The extrinsics $R, \mathbf{t}$ move the camera, and the perspective changes.

In [ ]:
C = np.array([12.0, 9.0, -25.0])
R, t = look_at(C, [-3, 5, 30])
street = street_scene()
def K_of(ax, ay, px, py):
    return np.array([[ax, 0, px], [0, ay, py], [0, 0, 1.0]])
phases = np.linspace(0, 2 * np.pi, 40, endpoint=False)
frames = [label(photo(street, K_of(f, f, 160, 120), R, t, 320, 240), f'f = {f:.0f} px', scale=0.5)
          for f in 250 + 120 * np.sin(phases)]
save_gif(frames, 'camera_intr_f', duration=90)
frames = [label(photo(street, K_of(250, 250, 160 + 60 * np.cos(a), 120 + 40 * np.sin(a)), R, t, 320, 240),
                f'p = ({160 + 60 * np.cos(a):.0f}, {120 + 40 * np.sin(a):.0f})', scale=0.5) for a in phases]
save_gif(frames, 'camera_intr_p', duration=90)
frames = [label(photo(street, K_of(250, 250 * r, 160, 120), R, t, 320, 240), f'alpha_y / alpha_x = {r:.2f}', scale=0.5)
          for r in 1 + 0.35 * np.sin(phases)]
save_gif(frames, 'camera_intr_m', duration=90)

In [ ]:
# extrinsics: the camera circles the street (always looking at its middle); left: seen from above
target = np.array([0.0, 5.0, 40.0])
K = intrinsics(250, 160, 120)
frames = []
for a in np.radians(np.concatenate([np.linspace(-70, 70, 40), np.linspace(70, -70, 40)])):
    C = target + 75 * np.array([np.sin(a), 0, -np.cos(a)]) + [0, 20, 0]
    R, t = look_at(C, target)
    image = photo(street, K, R, t, 320, 240, fog=1000)
    fig, ax = plt.subplots(figsize=(2.4, 2.4), dpi=100)
    for x0 in (-X_FACE - BLDG[0], X_FACE):
        for z0 in range(0, 90, 15):
            ax.add_patch(plt.Rectangle((x0, z0), BLDG[0], BLDG[2], color='0.6'))
    for corner in ([0, 0], [320, 0]):                  # the frustum: the rays of two image corners
        ray = R.T @ np.linalg.solve(K, [corner[0], 120, 1.0])
        ax.plot([C[0], C[0] + 120 * ray[0] / np.hypot(ray[0], ray[2])], [C[2], C[2] + 120 * ray[2] / np.hypot(ray[0], ray[2])], 'C3-', lw=1)
    ax.plot(C[0], C[2], 'C3o')
    ax.set_xlim(85, -85)                               # x points left when looking down with z up the page
    ax.set_ylim(-45, 125)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('from above', fontsize=9)
    fig.tight_layout(pad=0.1)
    fig.canvas.draw()
    top = cv2.resize(np.asarray(fig.canvas.buffer_rgba())[..., :3], (240, 240))
    plt.close(fig)
    frames.append(np.concatenate([top, np.full((240, 10, 3), 255, np.uint8), label(image, 'camera image', scale=0.5)], 1))
save_gif(frames, 'camera_extrinsics', duration=80)

## Radial lens distortion

Real lenses bend rays more (or less) the further they are from the optical axis. The simplest model moves each normalised image point radially: $\mathbf{x}_d = \mathbf{x}\,(1 + k_1 r^2)$ with $r = \|\mathbf{x}\|$. With $k_1 < 0$ (*barrel*) straight lines bow outwards, with $k_1 > 0$ (*pincushion*) inwards. Distortion is not linear in homogeneous coordinates, so it is not part of $P$: calibration estimates $k_1$ separately (below).

In [ ]:
C = np.array([3.0, 1.7, -14.0])
R, t = look_at(C, [-2, 4.0, 25])
K = intrinsics(260, 240, 180)
frames = []
for k1 in 0.25 * np.sin(np.linspace(0, 2 * np.pi, 50, endpoint=False)):
    kind = 'barrel' if k1 < -0.01 else 'pincushion' if k1 > 0.01 else 'none'
    frames.append(label(photo(street, K, R, t, 480, 360, k1=k1), f'k1 = {k1:+.2f} ({kind})'))
save_gif(frames, 'camera_distortion', duration=90)

## Camera calibration

We photograph a **calibration object**: two checkerboards on two walls at a right angle. Its 3D points (the inner corners of the squares) are known *by construction*: we built it with 5 cm squares and put the world origin at the bottom of the edge between the walls. Their 2D projections are found in the photo by a corner detector (OpenCV). Our photo is rendered by the ray caster with a known camera, including barrel distortion, so we can check the answer.

**Linear method (DLT).** Each correspondence $\mathbf{x}_i \sim P \mathbf{X}_i$ gives two linear equations in the 12 entries of $P$. Stack them into $A \mathbf{p} = 0$ and take the right singular vector of the smallest singular value. Then split $P = K[R \mid \mathbf{t}]$: the left $3 \times 3$ block is $KR$, an upper-triangular times an orthogonal matrix, which is exactly an RQ decomposition.

In [ ]:
# calibration object: two checkerboards of 11 x 9 squares (5 cm) with a white margin, on the walls x = 0 and z = 0;
# the calibration points are their 10 x 8 inner corners, in metres
s = 0.05
grid = np.stack(np.meshgrid(np.arange(2, 12), np.arange(2, 10)), -1).reshape(-1, 2) * s
calib_points = np.concatenate([np.stack([np.zeros(len(grid)), grid[:, 1], grid[:, 0]], 1),
                               np.stack([grid[:, 0], grid[:, 1], np.zeros(len(grid))], 1)])
def rig_board(a, b, footprint=None):
    i, j = np.floor(a / s) - 1, np.floor(b / s) - 1
    inside = (i >= 0) & (i < 11) & (j >= 0) & (j < 9)
    return np.where(inside & ((i + j) % 2 == 0), 0.08, 0.92)[:, None] * np.ones(3)
table = lambda a, b, footprint: np.full((len(a), 3), 0.6)
rig = [('rect', np.array([-0.5, 0, -0.5]), np.array([0, 0, 3.0]), np.array([0, 2.0, 0]), BRICKS),   # the room
       ('rect', np.array([-0.5, 0, -0.5]), np.array([3.0, 0, 0]), np.array([0, 2.0, 0]), BRICKS),
       ('rect', np.array([-0.5, 0, -0.5]), np.array([3.0, 0, 0]), np.array([0, 0, 3.0]), table),
       ('rect', np.zeros(3), np.array([0, 0, 13 * s]), np.array([0, 11 * s, 0]), rig_board),     # the board on x = 0
       ('rect', np.zeros(3), np.array([13 * s, 0, 0]), np.array([0, 11 * s, 0]), rig_board)]     # the board on z = 0

# the true camera, including radial lens distortion (k1 < 0: barrel distortion), and its photo of the object
K_true = np.array([[800, 0, 490], [0, 790, 355], [0, 0, 1.0]])   # a 960x720 image
R_true, t_true = look_at(np.array([0.9, 0.62, 0.9]), [0.2, 0.27, 0.2])
k1_true = -0.3
calib_image = photo(rig, K_true, R_true, t_true, 960, 720, k1=k1_true, fog=None)

def project_distorted(X, K, R, t, k1):
    Xc = (R @ X.T + t[:, None]).T
    xn = Xc[:, :2] / Xc[:, 2:]                         # normalised image coordinates
    xd = xn * (1 + k1 * (xn ** 2).sum(1, keepdims=True))
    return xd @ K[:2, :2].T + K[:2, 2]

def dlt(X, x):
    A = []
    for (X_, Y_, Z_), (u, v) in zip(X, x):
        P = [X_, Y_, Z_, 1]
        A.append([*P, 0, 0, 0, 0, *(-u * np.array(P))])
        A.append([0, 0, 0, 0, *P, *(-v * np.array(P))])
    _, _, Vt = np.linalg.svd(np.array(A))
    return Vt[-1].reshape(3, 4)

# detect the corners: the edge between the walls runs down the middle of the photo, so we look for one
# board in each half (painting the other half white)
gray = cv2.cvtColor(calib_image, cv2.COLOR_RGB2GRAY)
grids = []
for side in (slice(None, 480), slice(480, None)):
    half = np.full_like(gray, 255)
    half[:, side] = gray[:, side]
    ok, corners = cv2.findChessboardCorners(half, (10, 8))
    assert ok, 'board not found'
    corners = cv2.cornerSubPix(half, corners, (5, 5), (-1, -1), (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-3))
    g = corners.reshape(8, 10, 2).astype(float)
    # OpenCV lists the corners row by row, but from which corner it starts depends on the image. We know how the
    # object stands: put the lowest row (y = 10 cm) first and start each row next to the edge between the walls
    if g[0, :, 1].mean() < g[-1, :, 1].mean():
        g = g[::-1]
    if abs(g[:, 0, 0].mean() - 480) > abs(g[:, -1, 0].mean() - 480):
        g = g[:, ::-1]
    grids.append(g)
observed = np.concatenate(grids).reshape(-1, 2)      # left half: the board on x = 0, right half: the board on z = 0
print(f'{len(observed)} corners detected')

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(calib_image)
ax.scatter(*observed.T, s=14, c='r')
origin = project_distorted(np.zeros((1, 3)), K_true, R_true, t_true, k1_true)[0]
for axis, name in zip(np.eye(3) * 0.12, 'XYZ'):        # the world axes (12 cm long) at the origin
    tip = project_distorted(axis[None], K_true, R_true, t_true, k1_true)[0]
    ax.annotate('', tip, origin, arrowprops=dict(arrowstyle='->', color='C0', lw=2.5))
    ax.text(*(tip + 0.25 * (tip - origin)), name, color='C0', fontsize=14, ha='center', va='center', weight='bold')
for i in (9, 79, 89, 159):                           # a few of the known 3D points (in cm)
    X_ = calib_points[i] * 100
    ax.annotate(f'({X_[0]:.0f}, {X_[1]:.0f}, {X_[2]:.0f})', observed[i], xytext=(0, -16), textcoords='offset points',
                ha='center', fontsize=11, bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.85))
ax.set_title('the calibration object: detected corners and their known 3D coordinates (cm)')
ax.axis('off')
savefig('camera_calib_rig')
plt.show()

def decompose(P):
    K, R = rq(P[:, :3])
    S = np.diag(np.sign(np.diag(K)))                 # make the diagonal of K positive
    K, R = K @ S, S @ R
    t = np.linalg.solve(K, P[:, 3])
    if np.linalg.det(R) < 0:                          # P is only defined up to sign
        R, t = -R, -t
    return K / K[2, 2], R, t

def reprojection_error(K, R, t, k1=0.0):
    # root mean square distance between reprojected and observed points (px)
    return np.sqrt(((project_distorted(calib_points, K, R, t, k1) - observed) ** 2).sum(1).mean())

K_lin, R_lin, t_lin = decompose(dlt(calib_points, observed))
print('true K:\n', K_true.round(1))
print('linear (DLT) K:\n', K_lin.round(1))
print(f'DLT reprojection error (RMS): {reprojection_error(K_lin, R_lin, t_lin):.2f} px')

**Non-linear method.** The DLT minimises an algebraic error and cannot model lens distortion. We refine all parameters (focal lengths, principal point, rotation, translation and the distortion coefficient $k_1$) by minimising the *reprojection error* directly, starting from the DLT solution.

In [ ]:
from scipy.spatial.transform import Rotation

def unpack(params):
    fx, fy, cx, cy, k1 = params[:5]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    return K, Rotation.from_rotvec(params[5:8]).as_matrix(), params[8:11], k1

def residuals(params, model_distortion):
    K, R, t, k1 = unpack(params)
    return (project_distorted(calib_points, K, R, t, k1 if model_distortion else 0.0) - observed).ravel()

x0 = np.concatenate([[K_lin[0, 0], K_lin[1, 1], K_lin[0, 2], K_lin[1, 2], 0.0], Rotation.from_matrix(R_lin).as_rotvec(), t_lin])
results = {'linear (DLT)': (K_lin, R_lin, t_lin, 0.0)}
for name, distortion in [('non-linear, no distortion', False), ('non-linear + radial distortion', True)]:
    K, R, t, k1 = unpack(least_squares(residuals, x0, args=(distortion,)).x)
    results[name] = (K, R, t, k1 if distortion else 0.0)

print(f'{"":32s}  fx     fy     cx     cy     k1     RMS error (px)')
print(f'{"true":32s} {K_true[0,0]:6.1f} {K_true[1,1]:6.1f} {K_true[0,2]:6.1f} {K_true[1,2]:6.1f} {k1_true:6.3f}')
for name, (K, R, t, k1) in results.items():
    print(f'{name:32s} {K[0,0]:6.1f} {K[1,1]:6.1f} {K[0,2]:6.1f} {K[1,2]:6.1f} {k1:6.3f}   {reprojection_error(K, R, t, k1):.2f}')

fig, axs = plt.subplots(1, len(results), figsize=(24, 6.5))
for ax, (name, (K, R, t, k1)) in zip(axs, results.items()):
    predicted = project_distorted(calib_points, K, R, t, k1)
    err = predicted - observed
    ax.imshow(calib_image // 2 + 127)                 # the photo, faded
    ax.scatter(*observed.T, s=10, c='k', label='detected corner')
    ax.scatter(*predicted.T, s=40, facecolors='none', edgecolors='r', label='reprojected with the estimate')
    ax.quiver(*observed.T, *(30 * err).T, color='r', angles='xy', scale_units='xy', scale=1, width=0.004)
    ax.set_title(f'{name}: RMS error {reprojection_error(K, R, t, k1):.2f} px (arrows x30)')
    ax.axis('off')
    ax.legend(loc='lower left')
savefig('camera_calibration')
if SAVE_FIGURES:   # each panel on its own, so that the slide can show them one by one
    fig.canvas.draw()
    for i, ax in enumerate(axs):
        box = ax.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
        fig.savefig(f'figures/camera_calibration_{i + 1}.png', dpi=150, bbox_inches=box.padded(0.1))
plt.show()

## Calibration in practice: a checkerboard and OpenCV

A 3D calibration object is awkward to build. **Zhang's method** (the one in OpenCV and Matlab) needs only a flat checkerboard photographed in several poses: each view of the plane gives a homography, each homography two constraints on $K$, and a non-linear refinement (with distortion) finishes, exactly as above. We render 8 views of a 9 x 6 board with our ray caster, with a known $K$ and barrel distortion, and let OpenCV find the corners and calibrate.

In [ ]:
square = 0.04                                          # 4 cm squares, 10 x 7 of them: 9 x 6 inner corners
def board(a, b, footprint=None):
    i, j = np.floor(a / square) - 1, np.floor(b / square) - 1
    inside = (i >= 0) & (i < 10) & (j >= 0) & (j < 7)
    return np.where(inside & ((i + j) % 2 == 0), 0.05, 0.95)[:, None] * np.ones(3)
calib_scene = [('rect', np.array([-2, -2, 0.3]), np.array([5.0, 0, 0]), np.array([0, 5.0, 0]), BRICKS),   # a wall behind
               ('rect', np.zeros(3), np.array([12 * square, 0, 0]), np.array([0, 9 * square, 0]), board)]
K_cam, k1_cam = np.array([[700, 0, 330], [0, 700, 235], [0, 0, 1.0]]), -0.3
centre = np.array([6 * square, 4.5 * square, 0])
rng = np.random.default_rng(1)
views, poses = [], []
for i in range(8):                                     # the camera in front of the board, from different angles
    C = centre + [rng.uniform(-0.3, 0.3), rng.uniform(-0.25, 0.25), -rng.uniform(0.6, 0.8)]
    R, t = look_at(C, centre + rng.uniform(-0.03, 0.03, 3), up=(rng.uniform(-0.3, 0.3), 1, 0))
    views.append(photo(calib_scene, K_cam, R, t, 640, 480, k1=k1_cam, fog=None, ss=3))

pattern = (9, 6)
object_points = np.zeros((9 * 6, 3), np.float32)
object_points[:, :2] = (np.mgrid[0:9, 0:6].T.reshape(-1, 2) + 2) * square   # the inner corners
found = []
for image in views:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    ok, corners = cv2.findChessboardCorners(gray, pattern)
    if ok:
        corners = cv2.cornerSubPix(gray, corners, (5, 5), (-1, -1), (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-3))
    found.append(corners if ok else None)
used = [i for i, c in enumerate(found) if c is not None]
flags = cv2.CALIB_FIX_K2 | cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST     # our lens has only k1
rms, K_est, dist, _, _ = cv2.calibrateCamera([object_points] * len(used), [found[i] for i in used], (640, 480), None, None, flags=flags)
print(f'corners found in {len(used)} of {len(views)} views')
print('true K:\n', K_cam, f'\nk1 = {k1_cam}')
print('estimated K:\n', K_est.round(1), f'\nk1 = {dist[0, 0]:.3f}, RMS reprojection error {rms:.2f} px')

fig, axs = plt.subplots(1, 4, figsize=(24, 5))
for ax, i in zip(axs[:3], used[:3]):
    shown = views[i].copy()
    cv2.drawChessboardCorners(shown, pattern, found[i], True)
    ax.imshow(shown)
    ax.set_title(f'view {i + 1}: corners found')
undistorted = cv2.undistort(views[used[0]], K_est, dist)
axs[3].imshow(undistorted)
axs[3].set_title('view 1 undistorted with the estimate')
for ax in axs:
    ax.axis('off')
savefig('camera_zhang')
plt.show()